# Task 1: Exploratory Data Analysis (EDA)
**Dataset:** ChEMBL Bioactivity Dataset — 6,843 compounds  
**Target:** Acetylcholinesterase (CHEMBL220) inhibitors  

This notebook covers:
- Basic statistics for pIC50 (mean, min, max, std)
- Histogram and density plot of pIC50
- Bar plot of bioactivity class counts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✅')

## 1.1 Load Dataset

In [ ]:
df = pd.read_csv('../data/bioactivity_dataset.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 1.2 Compute pIC50
pIC50 = −log₁₀(IC50 in Molar) = 9 − log₁₀(IC50 in nM)

| pIC50 | Activity |
|-------|----------|
| ≥ 6.0 | **Active** |
| 5.0 – 6.0 | **Intermediate** |
| ≤ 5.0 | **Inactive** |

In [ ]:
# standard_value is IC50 in nM
df['standard_value'] = df['standard_value'].replace(0, 0.001)  # avoid log(0)
df['pIC50'] = (9 - np.log10(df['standard_value'])).round(4)
print('pIC50 computed successfully ✅')
df[['chembl_id','smiles','standard_value','pIC50','bioactivity_class']].head()

## 1.3 Basic Statistics for pIC50

In [ ]:
stats = df['pIC50'].describe()

print('=' * 48)
print('         pIC50 Descriptive Statistics')
print('=' * 48)
print(f'  Count            : {stats["count"]:.0f}')
print(f'  Mean             : {stats["mean"]:.4f}')
print(f'  Standard Dev     : {stats["std"]:.4f}')
print(f'  Min              : {stats["min"]:.4f}')
print(f'  25th Percentile  : {stats["25%"]:.4f}')
print(f'  Median (50th)    : {stats["50%"]:.4f}')
print(f'  75th Percentile  : {stats["75%"]:.4f}')
print(f'  Max              : {stats["max"]:.4f}')
print('=' * 48)

print(f'\nActive (pIC50 ≥ 6):        {(df["pIC50"] >= 6).sum()} compounds')
print(f'Inactive (pIC50 ≤ 5):      {(df["pIC50"] <= 5).sum()} compounds')
print(f'Intermediate (5 < p < 6):  {((df["pIC50"] > 5) & (df["pIC50"] < 6)).sum()} compounds')

## 1.4 Histogram + Density Plot of pIC50

In [ ]:
colors = {'active': '#2ecc71', 'inactive': '#e74c3c', 'intermediate': '#f39c12'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('pIC50 Distribution — 6,843 ChEMBL Compounds', fontsize=15, fontweight='bold')

# Histogram
axes[0].hist(df['pIC50'], bins=50, color='#3498db', edgecolor='white', linewidth=0.5, alpha=0.85)
axes[0].axvline(df['pIC50'].mean(), color='red', linestyle='--', lw=2,
                label=f"Mean = {df['pIC50'].mean():.2f}")
axes[0].axvline(df['pIC50'].median(), color='orange', linestyle='--', lw=2,
                label=f"Median = {df['pIC50'].median():.2f}")
for sd, col in [(1,'purple'), (-1,'purple')]:
    axes[0].axvline(df['pIC50'].mean() + sd*df['pIC50'].std(), color=col,
                    linestyle=':', lw=1.5, label=f'{sd:+d} SD = {df["pIC50"].mean()+sd*df["pIC50"].std():.2f}' if sd==1 else '')
axes[0].set_xlabel('pIC50', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Histogram of pIC50', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# KDE
for cls, grp in df.groupby('bioactivity_class'):
    sns.kdeplot(grp['pIC50'], ax=axes[1], label=f'{cls} (n={len(grp)})',
                color=colors[cls], fill=True, alpha=0.30, linewidth=2.5)
axes[1].axvline(6.0, color='black', linestyle='--', lw=1.5, label='Active threshold (6.0)')
axes[1].axvline(5.0, color='gray',  linestyle='--', lw=1.5, label='Inactive threshold (5.0)')
axes[1].set_xlabel('pIC50', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Density Plot by Bioactivity Class', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/task1_pIC50_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

## 1.5 Bar Plot — Bioactivity Class Counts

In [ ]:
counts = df['bioactivity_class'].value_counts().reindex(['active','inactive','intermediate'])
print('Bioactivity class counts:')
for cls, v in counts.items():
    print(f'  {cls:15s}: {v:5d}  ({v/len(df)*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Bioactivity Class Distribution', fontsize=15, fontweight='bold')

bars = axes[0].bar(counts.index, counts.values,
                   color=[colors[c] for c in counts.index],
                   edgecolor='white', linewidth=1.5, width=0.55)
for bar, (cls, v) in zip(bars, counts.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Bioactivity Class', fontsize=12)
axes[0].set_ylabel('Number of Compounds', fontsize=12)
axes[0].set_title('Compound Count per Class', fontsize=13)
axes[0].set_ylim(0, counts.max() * 1.25)
axes[0].grid(axis='y', alpha=0.3)

wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=[colors[c] for c in counts.index], startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2})
for at in autotexts: at.set_fontweight('bold')
axes[1].set_title('Class Distribution (%)', fontsize=13)

plt.tight_layout()
plt.savefig('../figures/task1_bioactivity_counts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved ✅')

## 1.6 Box + Violin Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('pIC50 Distribution by Bioactivity Class', fontsize=14, fontweight='bold')
order = ['active', 'intermediate', 'inactive']

for cls in order:
    sub = df[df['bioactivity_class'] == cls]['pIC50']
    axes[0].boxplot(sub, positions=[order.index(cls)], widths=0.45,
                    patch_artist=True,
                    boxprops=dict(facecolor=colors[cls], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', markersize=1.5, alpha=0.3))
    axes[1].violinplot(sub, positions=[order.index(cls)], widths=0.7)

for ax in axes:
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(order)
    ax.set_ylabel('pIC50', fontsize=12)
    ax.axhline(6.0, color='black', linestyle='--', lw=1.2, alpha=0.5)
    ax.axhline(5.0, color='gray',  linestyle='--', lw=1.2, alpha=0.5)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_title('Box Plot', fontsize=12)
axes[1].set_title('Violin Plot', fontsize=12)

plt.tight_layout()
plt.savefig('../figures/task1_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Task 1 Complete!')